# Research workflow — offline: question to pinned evidence

Owner: repository founder. Current workflow: 2026-09-20.

For a QR, this notebook follows the [research journey](README.md#research-activities-and-journey):
**question → data selection → quality and timing → exploratory feature → comparison → backtest → candidate handoff**.
The executable path covers current dataset access and a small notebook prototype.
Later stages specify the next research decisions and remain **planned** in the
[roadmap](../docs/roadmap/README.md); they are not simulated platform capabilities.

Restart the kernel and **Run All** using the [operations guide](notebook-operations.md).
The default uses tiny synthetic saved trades and an OKX L2 book, makes no provider
requests and needs no existing acquisition or credentials. A supplied saved selection
is opened explicitly, without falling back to synthetic data if it fails.

Infrastructure failure experiments live in the
[engineer's diagnostic notebook](infrastructure-diagnostics.ipynb).
This walkthrough supplies no empirical alpha, fill model or frozen P1 acceptance.

Use the [live version](research-foundations-live.ipynb) to initiate a new bounded public capture.
This offline version makes no provider request.


## 1. Frame the research question

Specify the observable, universe, decision horizon and what would invalidate a conclusion.
This example asks whether the selected input can support a visible-depth diagnostic.
There is no fitted model, performance comparison or final-holdout inspection.
For an actual study, retain the hypothesis and every influencing attempt using the
[research record contract](../docs/roadmap/research-program.md#small-records-and-artifacts).


In [1]:
question = {
    "question": "Can the selected book support a visible-depth imbalance diagnostic?",
    "sample": "synthetic BTC/ETH trades and OKX BTC-USDT depth unless explicitly overridden",
    "observable": "visible bid and ask base quantities at one received book state",
    "horizon": "single-state inspection; a forward outcome needs separately evidenced prices",
    "comparator": "zero visible-depth imbalance; no predictive or economic claim",
    "stop_if": "required timing, units, fields or historical eligibility are unknown",
    "fitting": "not applicable to this notebook prototype",
    "holdout": "not opened; no formal comparison is performed",
}
question

{'question': 'Can the selected book support a visible-depth imbalance diagnostic?',
 'sample': 'synthetic BTC/ETH trades and OKX BTC-USDT depth unless explicitly overridden',
 'observable': 'visible bid and ask base quantities at one received book state',
 'horizon': 'single-state inspection; a forward outcome needs separately evidenced prices',
 'comparator': 'zero visible-depth imbalance; no predictive or economic claim',
 'stop_if': 'required timing, units, fields or historical eligibility are unknown',
 'fitting': 'not applicable to this notebook prototype',
 'holdout': 'not opened; no formal comparison is performed'}

### Prepare the installed candidate

The complete runtime file inventory must match the checkout. A package version alone
cannot identify changed source. This check records the interpreter, lock, source digest
and notebook source digest; the runner additionally preserves each execution attempt.
Expand setup cells if investigating an installation mismatch.


In [2]:
# standard library imports
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
from decimal import Decimal
from importlib.metadata import distributions
from pathlib import Path
from tempfile import TemporaryDirectory

# third party library imports
import pandas as pd
from IPython.display import display
from msgspec.structs import replace

# project library imports
import market_data
from applications.explore import inspect_eda
from market_data.adapters.sources import SourceAdapters
from market_data.contracts import Level
from market_data.datasets import SnapshotCatalog
from market_data.query import BookMarketRow, QueryMarketRow
from market_data.storage import RawStore
from research.exploration import (
    BookResearchInput,
    ResearchDataset,
    ResearchReader,
    ResearchSelection,
    TradeResearchInput,
    TradeSelection,
)
from runtime import ImplementationIdentity, RuntimeFiles
from serialization import DomainCodec

root = Path.cwd().resolve()
if root.name == "examples":
    root = root.parent
assert (root / "configs/data-jobs/binance-archive.json").is_file(), "Use the repository kernel/root"
assert platform.python_version() == (root / ".python-version").read_text().strip()
assert "site-packages" in Path(market_data.__file__).resolve().parts
pd.set_option("display.max_colwidth", None)
codec = DomainCodec()
workspace = TemporaryDirectory(prefix="qr-workflow-")
local = Path(workspace.name)

In [3]:
source_root = root / "src"
installed_inputs = dict(RuntimeFiles.inputs())
source_files = {
    path.relative_to(source_root).as_posix(): path
    for package in RuntimeFiles.roots
    for path in (source_root / package).rglob("*")
    if path.is_file() and (path.suffix in {".py", ".json"} or path.name == "py.typed")
}
assert source_files.keys() == installed_inputs.keys(), "Installed runtime inventory differs"
for name, source in source_files.items():
    assert source.read_bytes() == installed_inputs[name].read_bytes(), (
        f"Stale installed file: {name}"
    )
notebook_path = Path(
    os.environ.get("ALPHA_RESEARCH_NOTEBOOK_PATH", root / "examples/research-foundations.ipynb")
)
notebook = json.loads(notebook_path.read_text())
notebook_source = [(cell["cell_type"], "".join(cell["source"])) for cell in notebook["cells"]]
identity = {
    "commit": subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=root, text=True).strip(),
    "working_tree": "changed"
    if subprocess.check_output(["git", "status", "--short"], cwd=root, text=True).strip()
    else "clean",
    "python": platform.python_version(),
    "platform": platform.platform(),
    "tooling": {
        item.metadata["Name"]: item.version
        for item in distributions()
        if item.metadata["Name"] in {"nbclient", "nbformat", "ipykernel", "pandas"}
    },
    "kernel": sys.executable,
    "package": market_data.__file__,
    "runtime_files": len(installed_inputs),
    "implementation_sha256": ImplementationIdentity.sha256(),
    "lock_sha256": hashlib.sha256((root / "uv.lock").read_bytes()).hexdigest(),
    "notebook_source_sha256": hashlib.sha256(
        json.dumps(notebook_source, ensure_ascii=False, separators=(",", ":")).encode()
    ).hexdigest(),
    "attempt": os.environ.get("ALPHA_RESEARCH_ATTEMPT_ID", "interactive: unsnapshotted"),
}
display(pd.DataFrame(identity.items(), columns=["Identity", "Value"]))

,Identity,Value
0,commit,00373626acc1459136163297cd4a76d2c0542787
1,working_tree,changed
2,python,3.14.7
3,platform,macOS-27.0-arm64-arm-64bit-Mach-O
4,tooling,"{'nbformat': '5.10.4', 'ipykernel': '7.2.0', 'nbclient': '0.10.4', 'pandas': '3.0.5'}"
5,kernel,/Users/lxg/.cache/uv/builds-v0/.tmphUt7Bw/bin/python
6,package,/private/tmp/alpha-research-live-offline-notebooks/.venv/lib/python3.14/site-packages/market_data/__init__.py
7,runtime_files,138
8,implementation_sha256,b67f60a16bd5f84fc3b0898389638dca4e16a7372418fa5fd3b0ad80c8d9c5f9
9,lock_sha256,e6cd3a169ff4b6850bcb5b09e38c44e1179b0dc1fb7c6a9bd95d2c4a058595e4


## 2. Select or onboard the required data

For an existing study, set `ALPHA_RESEARCH_DATASET_DIRECTORY` to a retained directory
containing `raw/`, `snapshots/` and an authored `selection.json`. Both stores must retain
the bytes addressed by that selection. The notebook reads them and writes its own reports
to a separate temporary workspace. No mutable latest-input choice is made.

With no override, the collapsed setup calls the existing
[installed acquisition/publication consumer](../tests/integration/consumers/datasets/ingestion_workflow.py).
It supplies synthetic checksummed HTTP responses for the configured `spot-comparison`
job and a synthetic native book. Package acquisition and publication own the mechanics;
the notebook starts from their saved stores. This also exercises their failure paths,
whose engineering interpretation belongs in the diagnostic notebook.

To obtain new permitted data, use the existing [data commands](../src/market_data/README.md)
and [capture instructions](notebook-operations.md#acquire-and-reopen-permitted-inputs).
A download supplies bytes; it does not establish historical availability or eligibility.


In [4]:
override = os.environ.get("ALPHA_RESEARCH_DATASET_DIRECTORY")
synthetic = override is None
if synthetic:
    fixture_inputs = local / "inputs"
    fixture_inputs.mkdir()
    for relative in (
        "tests/integration/consumers/datasets/ingestion_workflow.py",
        "tests/support/subprocesses/interrupt_command.py",
        "tests/fixtures/eligibility/membership-corrections.json",
        "configs/data-jobs/binance-archive.json",
        "configs/live/okx-btc-usdt.json",
        "configs/reference/binance-archive-spot.json",
    ):
        source = root / relative
        shutil.copyfile(source, fixture_inputs / source.name)
    data_directory = local / "data"
    prepared = subprocess.run(
        [
            sys.executable,
            "-I",
            str(fixture_inputs / "ingestion_workflow.py"),
            str(fixture_inputs),
            str(data_directory),
        ],
        cwd=local,
        capture_output=True,
        text=True,
        timeout=60,
        check=False,
    )
    assert prepared.returncode == 0, (prepared.stdout, prepared.stderr)
else:
    data_directory = Path(override).expanduser().resolve()
if not all((data_directory / name).is_dir() for name in ("raw", "snapshots")):
    raise FileNotFoundError(f"Saved study requires raw/ and snapshots/: {data_directory}")
saved_selection = codec.decode_json(
    (data_directory / "selection.json").read_bytes(), ResearchSelection
)
print("Input mode:", "synthetic software example" if synthetic else "explicit saved selection")
print("Retained input directory:", data_directory)
print("Saved selection identity:", saved_selection.identity)

Input mode: synthetic software example
Retained input directory: /var/folders/30/8y6v8_j93_d8k3xzh930ghqw0000gn/T/qr-workflow-o50wgdvr/data
Saved selection identity: afb7abf0012d497ec91bd22540cdcb26e5adc64078990fe647c2a1c22a1470a0


### Discover metadata, then pin the study

Discovery is bounded metadata inspection. Opening the selection later verifies all
required raw and canonical inputs. Choose the named bindings, immutable snapshot IDs,
fields, intervals and one shared inclusive knowledge cutoff explicitly. Trade intervals
use business time; book intervals use receipt time; both exclude the end.

Edit `selection` below by constructing or replacing the public selection records, using
[the reference](../src/market_data/README.md#discover-and-reopen-research-selections).
Keeping the loaded records is an explicit pin to their saved identities, not to the
newest snapshot. Change byte/entry caps only for the intended workload; batch size does
not cap total process memory.


In [5]:
candidates = SnapshotCatalog(data_directory / "snapshots").discover(
    max_entries=64, max_bytes=2_097_152
)
display(
    pd.DataFrame(
        [
            {
                "snapshot": item.identity,
                "family": type(item).__name__,
                "partitions": len(item.partitions),
                "quality_scope": item.scope,
            }
            for item in candidates
        ]
    )
)
selection = ResearchSelection(inputs=saved_selection.inputs)
selection_path = local / "selection.json"
selection_path.write_bytes(codec.encode_json(selection))
display(
    pd.DataFrame(
        [
            {
                "name": item.name,
                "family": "trade" if isinstance(item, TradeSelection) else "book",
                "snapshot": item.query.snapshot_id,
                "fields": item.query.columns,
                "instruments": tuple(map(str, item.query.instruments))
                if isinstance(item, TradeSelection)
                else (str(item.query.instrument),),
                "start": item.query.start,
                "end_excluded": item.query.end,
                "known_at_inclusive": item.query.known_at,
                "byte_cap": item.query.max_bytes,
            }
            for item in selection.inputs
        ]
    )
)
print("Pinned selection:", selection.identity)

,snapshot,family,partitions,quality_scope
0,5a2024727d12a66228d7f6ff0ffc6b1344374422feb55c8e707ae1f16bfdc14d,BookDatasetManifest,1,exploratory_degraded
1,84df4bdcf639d98c03d057d9b4e5039a3b81eb86ff3e3cd80946715809112bd2,DatasetManifest,2,exploratory_degraded


,name,family,snapshot,fields,instruments,start,end_excluded,known_at_inclusive,byte_cap
0,btc,trade,84df4bdcf639d98c03d057d9b4e5039a3b81eb86ff3e3cd80946715809112bd2,"(price, revision)","(11111111-1111-4111-8111-111111111111,)",2024-06-01 00:00:00+00:00,2024-06-01 01:00:00+00:00,2026-09-18 00:00:00+00:00,262144
1,eth,trade,84df4bdcf639d98c03d057d9b4e5039a3b81eb86ff3e3cd80946715809112bd2,"(price, revision)","(22222222-2222-4222-8222-222222222222,)",2024-06-01 00:00:00+00:00,2024-06-01 01:00:00+00:00,2026-09-18 00:00:00+00:00,262144
2,okx_depth,book,5a2024727d12a66228d7f6ff0ffc6b1344374422feb55c8e707ae1f16bfdc14d,"(bids, asks)","(4476df8e-6342-4417-839e-e41b88f41446,)",2024-06-01 00:00:00+00:00,2024-06-01 01:00:00+00:00,2026-09-18 00:00:00+00:00,262144


Pinned selection: afb7abf0012d497ec91bd22540cdcb26e5adc64078990fe647c2a1c22a1470a0


## 3. Inspect values, missingness and clocks together

The existing EDA entry retains started, completed and failed records through a caller-owned
callback. It returns separate typed inputs, without implicit joins, currency conversion or
forward filling. Inspect coverage and control rows even when no market values are selected.
Unknown archive availability should produce unavailable values, never fabricated timestamps.


In [6]:
class StudyReports:
    """Append the small EDA attempt records in this notebook's private workspace."""

    @staticmethod
    def retain(record: dict[str, object]) -> None:
        """Save one started, completed or failed application record.

        Args:
            record: Application-owned identity, status and evidence.
        """
        with (local / "eda-attempts.jsonl").open("ab") as output:
            output.write(codec.encode_json(record) + b"\n")


result = inspect_eda(
    selection_path,
    data_directory / "snapshots",
    RawStore(data_directory / "raw"),
    report=StudyReports.retain,
)
dataset = result["dataset"]
assert isinstance(dataset, ResearchDataset) and dataset.identity == selection.identity
observations = []
for item in dataset.inputs:
    if isinstance(item, TradeResearchInput):
        evidence = item.batches[0].evidence
        count = sum(isinstance(row, QueryMarketRow) for batch in item.batches for row in batch.rows)
        units = tuple(str(part.request.instrument.quote_currency.code) for part in evidence)
        reasons = sorted({reason for coverage in item.coverage for reason in coverage.reasons})
    else:
        count = sum(isinstance(row, BookMarketRow) for batch in item.batches for row in batch.rows)
        config = item.batches[0].evidence.request.config
        units = (config.quote_currency,)
        reasons = sorted({reason for batch in item.batches for reason in batch.reasons})
    total = sum(len(batch.rows) for batch in item.batches)
    observations.append(
        {
            "input": item.name,
            "market_rows": count,
            "controls": total - count,
            "quote_currency": units,
            "reasons": reasons,
        }
    )
display(pd.DataFrame(observations))
if synthetic:
    assert [row["market_rows"] for row in observations] == [0, 0, 1]

,input,market_rows,controls,quote_currency,reasons
0,btc,0,0,"(USDT,)","[catalog units are authored research assumptions, no selected trade values; inspect availability and membership evidence, observed values do not establish complete coverage, original trade availability unknown, recorder coverage unknown]"
1,eth,0,0,"(USDT,)","[catalog units are authored research assumptions, no selected trade values; inspect availability and membership evidence, observed values do not establish complete coverage, original trade availability unknown, recorder coverage unknown]"
2,okx_depth,1,3,"(USDT,)",[visible depth only; coverage and historical eligibility unproven]


The default trades contain six retained historical observations but **zero decision-eligible
trade values**: archive download time is not original availability. The one book market
row has an actual synthetic receipt coordinate. Controls remain separate from values.
Coverage and historical eligibility remain unproven even for the observed book state.

**Exercise:** move the common cutoff to the beginning of the selected interval. Which
values can still be available? For the default sample, the snapshot arrives one
microsecond after the interval begins; it cannot appear early. The following cell is an
answer scaffold that changes the query, not the saved evidence.


In [7]:
earlier_cutoff = min(item.query.start for item in selection.inputs)
earlier_selection = ResearchSelection(
    inputs=tuple(
        replace(item, query=replace(item.query, known_at=earlier_cutoff))
        for item in selection.inputs
    )
)
reader = ResearchReader(
    data_directory / "snapshots", RawStore(data_directory / "raw"), readers=SourceAdapters()
)
earlier = reader.read(earlier_selection)
earlier_counts = [
    (
        item.name,
        sum(
            isinstance(row, (BookMarketRow, QueryMarketRow))
            for batch in item.batches
            for row in batch.rows
        ),
    )
    for item in earlier.inputs
]
display(pd.DataFrame(earlier_counts, columns=["Input", "Earlier market rows"]))
if synthetic:
    assert all(count == 0 for _, count in earlier_counts)
assert reader.read(selection) == dataset, "Earlier inspection changed the pinned result"
print("Earlier selection:", earlier.identity)
print("Original selection reproduced:", dataset.identity)

,Input,Earlier market rows
0,btc,0
1,eth,0
2,okx_depth,0


Earlier selection: 7955823cf3ae0f230c8b3884cc76c1c530396dda032d0c6cdaefaf57060751ff
Original selection reproduced: afb7abf0012d497ec91bd22540cdcb26e5adc64078990fe647c2a1c22a1470a0


Corrections and historical membership also depend on when they became known. A final
revised value must not be used at every earlier decision. The
[installed cutoff example](../tests/integration/consumers/query/asof_consumer.py) retains
an unknown original, a correction available at second 10 and a membership withdrawal at
noon; [access regressions](../tests/unit/data/test_research_access.py) cover those boundaries.
The engineer notebook runs the existing cutoff consumer. Changing a query cannot repair
missing source evidence.


## 4. Prototype a feature and specify its label separately

**Available as notebook exploration; shared versioned feature/label workflows are planned.**
For the default synthetic L2 state, calculate visible-size imbalance
`(bid BTC - ask BTC) / (bid BTC + ask BTC)`. This is dimensionless and uses only the two
selected sides at the row's original receipt. Keep decimal arithmetic and attach its
state/selection identity and degraded quality. A zero denominator means unavailable.

The calculation is deliberately restricted to the named synthetic teaching input.
For a saved study, first supply the appropriate formula, units, stale-state and missingness
policy; the notebook will not apply an assumed BTC formula to arbitrary data.
A later return label needs a separately defined future horizon, price convention and
label-availability interval. It cannot enter an earlier feature vector.


In [8]:
prototype = []
if synthetic:
    depth = next(item for item in dataset.inputs if item.name == "okx_depth")
    assert isinstance(depth, BookResearchInput)
    for batch in depth.batches:
        assert batch.columns == ("bids", "asks")
        for row in batch.rows:
            if not isinstance(row, BookMarketRow):
                continue
            sizes = []
            for side in row.values:
                assert all(
                    isinstance(level, tuple) and isinstance(level[1], Level) for level in side
                )
                sizes.append(sum((level[1].size for level in side), Decimal(0)))
            total = sizes[0] + sizes[1]
            imbalance = None if total == 0 else (sizes[0] - sizes[1]) / total
            prototype.append(
                {
                    "received_ns": row.received_ns,
                    "state": row.state_identity,
                    "bid_BTC": str(sizes[0]),
                    "ask_BTC": str(sizes[1]),
                    "imbalance": str(imbalance),
                    "quality": row.status,
                }
            )
    assert len(prototype) == 1 and prototype[0]["imbalance"] == "-0.2"
    display(pd.DataFrame(prototype))
else:
    print("Prototype not applied: define a study-specific formula and unit/quality policy first.")

,received_ns,state,bid_BTC,ask_BTC,imbalance,quality
0,1717200000000001000,0027d937a22fbbf77271305be379fc7a38cacdc4520171e1d33c0c7955ccd740,2,3,-0.2,degraded


One synthetic state verifies arithmetic and provenance; it supplies no distribution,
predictive value or tradeable edge. Controls are never feature observations. In a real
study, inspect coverage, warm-up, stale intervals and stability before comparison, and
retain attempts that influence selection. Promote a transform into the supported package
interface when formal retained consumers need it; keep the notebook useful for exploration.

## 5. Compare models or deterministic rules — planned

Next supply a transparent rule or an optional estimator, time-ordered splits, information
intervals, seeds, selection metrics and stopping criteria. Fit preprocessing only within
training data; keep future labels and consumed holdouts out of new selection. A rule can
record fitting as inapplicable. The shared attempt/split and fit workflows are still
[roadmap work](../docs/roadmap/README.md#experiment-attempts-and-temporal-splits).
No score or fitted artifact is fabricated here.

## 6. Backtest and evaluate economics — planned

A signal or prediction needs a separate signal-to-target policy, independent limits,
execution timing, costs, fills and accounting. Reconcile a tiny economic trace before
interpreting net returns. Current book/trade replay establishes input behavior, not
executable liquidity, queue position or P&L. The
[first economic backtest](../docs/roadmap/README.md#first-economic-backtest) remains planned.

## 7. Freeze, qualify and monitor — planned

An accepted candidate will pin dataset, feature, optional fitted model, policy and
experiment evidence. Replay/paper qualification, deployment and actual activation are
separate steps. Collector health diagnostics do not establish strategy health. The
[research guide](README.md#freeze-and-qualify-a-candidate) describes the handoff;
no candidate, deployment or trading activity is created by running this notebook.

### Retain the conclusion and next action

For the default sample, retain a software-only conclusion: named inputs reopen, unknown
trade availability remains visible, an early cutoff cannot see a later book state, and
one exact exploratory diagnostic can be traced to its input. Historical eligibility,
coverage and a future label remain missing for empirical research.

The cell below prepares a compact handoff, not the future experiment journal. Saved
study bytes remain in the explicitly supplied directory. Synthetic stores and temporary
reports are deleted at the end; their hashes alone do not make those deleted bytes
retrievable. The runner retains successful/failed notebook snapshots locally, while Git
retains reviewed source and small outputs. Preserve real influencing attempts and data
in the protected research record before replacing conclusions.


In [9]:
handoff = {
    "question": question["question"],
    "selection_identity": dataset.identity,
    "input_snapshots": tuple(item.query.snapshot_id for item in selection.inputs),
    "implementation_identity": identity["implementation_sha256"],
    "sample_kind": "synthetic"
    if synthetic
    else "explicit saved study; inspect retained qualifications",
    "eligible_market_rows": {row["input"]: row["market_rows"] for row in observations},
    "next_action": "resolve applicable source/label evidence before formal comparison",
    "formal_comparison": "not performed",
    "backtest": "not implemented",
}
display(pd.DataFrame(handoff.items(), columns=["Handoff", "Value"]))
workspace.cleanup()

,Handoff,Value
0,question,Can the selected book support a visible-depth imbalance diagnostic?
1,selection_identity,afb7abf0012d497ec91bd22540cdcb26e5adc64078990fe647c2a1c22a1470a0
2,input_snapshots,"(84df4bdcf639d98c03d057d9b4e5039a3b81eb86ff3e3cd80946715809112bd2, 84df4bdcf639d98c03d057d9b4e5039a3b81eb86ff3e3cd80946715809112bd2, 5a2024727d12a66228d7f6ff0ffc6b1344374422feb55c8e707ae1f16bfdc14d)"
3,implementation_identity,b67f60a16bd5f84fc3b0898389638dca4e16a7372418fa5fd3b0ad80c8d9c5f9
4,sample_kind,synthetic
5,eligible_market_rows,"{'btc': 0, 'eth': 0, 'okx_depth': 1}"
6,next_action,resolve applicable source/label evidence before formal comparison
7,formal_comparison,not performed
8,backtest,not implemented
